# Dual-Stream Model: Simultaneous Text Classification Test

This notebook tests whether the dual-stream model can simultaneously classify two different texts into topics using separate output heads.


In [1]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer
import json
import os


## Configuration


In [2]:
# Checkpoint path (easily changeable)
CHECKPOINT_PATH = "checkpoints/run_20251224_185822/checkpoint-160000/model.pt"

device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda:1


## Load Model and Tokenizer


In [3]:
def find_split_token_index(encoding, split_char_idx):
    """
    Находит индекс токена, с которого начинается суффикс,
    основываясь на позиции символа разрыва в тексте.
    
    Принцип: Вместо поиска по тексту суффикса используем offset_mapping
    токенизатора, чтобы найти первый токен, который начинается >= split_char_idx.
    
    Args:
        encoding: результат tokenizer() с return_offsets_mapping=True
        split_char_idx: позиция символа в исходном тексте, где начинается суффикс
    
    Returns:
        int: индекс токена (используется для обрезки: input_ids[:idx])
             Если суффикс пустой или начинается в конце, вернет длину последовательности.
    """
    offsets = encoding.offset_mapping[0]  # shape (seq_len, 2)
    input_ids = encoding.input_ids[0]
    
    found_idx = len(input_ids)  # По умолчанию - конец (для пустого суффикса)
    
    for idx, (start, end) in enumerate(offsets):
        # Пропускаем специальные токены (у них часто offset (0,0))
        if start == end:
            continue
        
        # Сценарий 1: Чистая граница. Токен начинается ровно там или дальше суффикса.
        if start >= split_char_idx:
            found_idx = idx
            break
        
        # Сценарий 2 (Сложный): "Straddling token".
        # Граница разрыва попала внутрь токена (start < split < end).
        # Это значит, что конец Body и начало Suffix слились в один токен.
        if start < split_char_idx < end:
            # Conservative подход: обрезать ЭТО токен.
            # Мы потеряем пару букв конца Body, но зато Suffix
            # будет чистым и идентичным для обоих потоков.
            found_idx = idx
            break
    
    return found_idx

print("✓ Helper function find_split_token_index defined")
print()


✓ Helper function find_split_token_index defined



In [ ]:
from model import DualStreamLLM

# Load checkpoint metadata to get model configuration
checkpoint_dir = os.path.dirname(CHECKPOINT_PATH)
config_path = os.path.join(checkpoint_dir, "../training_config.json")

with open(config_path, 'r') as f:
    training_config = json.load(f)

print("Training Configuration:")
print(f"  Checkpoint path: {checkpoint_dir}")
print(f"  Config Path: {config_path}")
print(f"  Model: {training_config['model_name']}")
print(f"  Objective: {training_config['objective']}")
print(f"  Shared embeddings: {training_config['shared_embeddings']}")
print(f"  Embedding init: {training_config['embedding_init_strategy']}")
print()


Training Configuration:
  Checkpoint path: checkpoints/run_20251224_185822/checkpoint-160000
  Config Path: checkpoints/run_20251224_185822/checkpoint-160000/../training_config.json
  Model: EleutherAI/pythia-2.8b
  Objective: separate
  Shared embeddings: True
  Embedding init: copy



In [5]:
# Initialize model with same configuration as during training
model = DualStreamLLM(
    model_name=training_config['model_name'],
    freeze_backbone=training_config['freeze_backbone'],
    embedding_init_strategy=training_config['embedding_init_strategy'],
    unembedding_init_strategy=training_config.get('unembedding_init_strategy'),
    embedding_init_noise_std=training_config['embedding_init_noise_std'],
    objective=training_config['objective'],
    shared_embeddings=training_config['shared_embeddings']
)

# Load checkpoint weights
print(f"Loading checkpoint from: {CHECKPOINT_PATH}")
checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu')

# Load model state
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)
model.eval()

print(f"✓ Model loaded successfully from step {checkpoint['step']}")
print()


Loading checkpoint from: checkpoints/run_20251224_185822/checkpoint-160000/model.pt


/tmp/ipykernel_1414928/273160220.py:14: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu')


✓ Model loaded successfully from step 160000



In [6]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(training_config['model_name'])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


print(f"✓ Tokenizer loaded: {training_config['model_name']}")
print(f"  Vocab size: {len(tokenizer)}")
print()


✓ Tokenizer loaded: EleutherAI/pythia-2.8b
  Vocab size: 50277



## Prepare Test Texts for Dual Classification


In [7]:
# prefix = "Possible Categories: Art, Business, Cooking, History, Music, Politics, Science, Sports, Technology, Travel.\n\n"
# suffix = '\nText Category: "'
# body_a = "Text: The astronomers observed a massive black hole in a distant galaxy."
# body_b = "Text: The striker kicked the soccer ball into the net to win the match."


prefix = ""
suffix = ''
body_a = "300 400 500 600 700 800"
body_b = "1 2 3 4 5 6 7 8 9 10 11"



# Construct full strings
text_a = prefix + body_a + suffix
text_b = prefix + body_b + suffix

print("Text A:")
print(text_a)
print('-' * 30)
print("\nText B:")
print(text_b)
print()


Text A:
300 400 500 600 700 800
------------------------------

Text B:
1 2 3 4 5 6 7 8 9 10 11



## Tokenize and Align Sequences


In [8]:

# Tokenize full strings with offset mapping
# return_offsets_mapping gives us the (start, end) char index for each token
enc_a = tokenizer(text_a, return_offsets_mapping=True, return_tensors="pt")
enc_b = tokenizer(text_b, return_offsets_mapping=True, return_tensors="pt")

# Find the character position where suffix begins
# Using length-based approach instead of text search (Способ 1)
boundary_idx_a = len(prefix) + len(body_a)
boundary_idx_b = len(prefix) + len(body_b)

# Map character index to token index using offset_mapping
# This handles empty suffixes and tokenization boundary effects correctly
split_token_idx_a = find_split_token_index(enc_a, boundary_idx_a)
split_token_idx_b = find_split_token_index(enc_b, boundary_idx_b)

print(f"Boundary at char {boundary_idx_a} -> token {split_token_idx_a} for A")
print(f"Boundary at char {boundary_idx_b} -> token {split_token_idx_b} for B")
print(f"Full sequences: A={enc_a.input_ids.shape[1]} tokens, B={enc_b.input_ids.shape[1]} tokens")

# Determine truncation length
# We take the minimum length of the content BEFORE the suffix
# to ensure both streams are aligned at the suffix boundary
min_content_len = min(split_token_idx_a, split_token_idx_b)

print(f"Truncating to {min_content_len} tokens before suffix")
print()


Boundary at char 23 -> token 6 for A
Boundary at char 23 -> token 11 for B
Full sequences: A=6 tokens, B=11 tokens
Truncating to 6 tokens before suffix



In [ ]:
# Construct aligned tensors
# We take the pre-calculated token IDs from the FULL tokenization (preserving context merges)
# and slice them up to the cut-off point
context_a = enc_a.input_ids[0, :min_content_len]
context_b = enc_b.input_ids[0, :min_content_len]

# Prepare suffix tokens
# Since we know the suffix starts with \n, it forms a hard boundary
# We tokenize it separately to ensure it is identical for both streams
suffix_ids = tokenizer(suffix, add_special_tokens=False, return_tensors="pt").input_ids[0]

# Concatenate truncated context with the uniform suffix
input_a = torch.cat([context_a, suffix_ids]).unsqueeze(0).to(device)
input_b = torch.cat([context_b, suffix_ids]).unsqueeze(0).to(device)

# Create masks (all ones since we truncated)
mask_a = torch.ones_like(input_a)
mask_b = torch.ones_like(input_b)

print(f"Final aligned length: {input_a.shape[1]}")
print(f"Input A shape: {input_a.shape}")
print(f"Input B shape: {input_b.shape}")
print()


# Decode the aligned sequences
aligned_text_a = tokenizer.decode(input_a[0].cpu().tolist())
aligned_text_b = tokenizer.decode(input_b[0].cpu().tolist())

print(f"\nAligned Text A ({input_a.shape[1]} tokens):")
print(aligned_text_a)
print(f"\nAligned Text B ({input_b.shape[1]} tokens):")
print(aligned_text_b)


Final aligned length: 6
Input A shape: torch.Size([1, 6])
Input B shape: torch.Size([1, 6])


Aligned Text A (6 tokens):
300 400 500 600 700 800

Aligned Text B (6 tokens):
1 2 3 4 5 6


## Run Inference


In [10]:
# Run model generation (3 tokens)
model.eval()
generated_A, generated_B = model.generate(
    input_ids_A=input_a,
    attention_mask_A=mask_a,
    input_ids_B=input_b,
    attention_mask_B=mask_b,
    max_new_tokens=3,
    temperature=0.7,
    do_sample=True,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id
)

print(f"Generated A shape: {generated_A.shape}")
print(f"Generated B shape: {generated_B.shape}")
print()



Generated A shape: torch.Size([1, 9])
Generated B shape: torch.Size([1, 9])



## Decode Predictions


In [11]:
# Decode generated tokens (only the new tokens)
input_len = input_a.shape[1]
generated_tokens_a = generated_A[0, input_len:]
generated_tokens_b = generated_B[0, input_len:]

pred_text_a = tokenizer.decode(generated_tokens_a, skip_special_tokens=True)
pred_text_b = tokenizer.decode(generated_tokens_b, skip_special_tokens=True)

print("-" * 60)
print("DUAL CLASSIFICATION RESULTS")
print("-" * 60)
print(f"Head A Prediction: '{pred_text_a}'")
print(f"Head B Prediction:  '{pred_text_b}'")
print("-" * 60)


------------------------------------------------------------
DUAL CLASSIFICATION RESULTS
------------------------------------------------------------
Head A Prediction: ' 750 8 9'
Head B Prediction:  ' 7 8 9'
------------------------------------------------------------


## Test with Additional Examples


In [12]:
def test_dual_classification(body_a, body_b, expected_a, expected_b, prefix, suffix):
    """
    Test dual classification on two text bodies.
    
    Args:
        body_a: Text body for stream A
        body_b: Text body for stream B
        expected_a: Expected category for stream A
        expected_b: Expected category for stream B
    """
    # Construct full strings
    text_a = prefix + body_a + suffix
    text_b = prefix + body_b + suffix
    
    # Tokenize
    enc_a = tokenizer(text_a, return_offsets_mapping=True, return_tensors="pt")
    enc_b = tokenizer(text_b, return_offsets_mapping=True, return_tensors="pt")
    
    # Find suffix positions using length-based approach
    boundary_idx_a = len(prefix) + len(body_a)
    boundary_idx_b = len(prefix) + len(body_b)
    split_token_idx_a = find_split_token_index(enc_a, boundary_idx_a)
    split_token_idx_b = find_split_token_index(enc_b, boundary_idx_b)
    
    # Align sequences
    min_content_len = min(split_token_idx_a, split_token_idx_b)
    context_a = enc_a.input_ids[0, :min_content_len]
    context_b = enc_b.input_ids[0, :min_content_len]
    suffix_ids = tokenizer(suffix, add_special_tokens=False, return_tensors="pt").input_ids[0]
    
    input_a = torch.cat([context_a, suffix_ids]).unsqueeze(0).to(device)
    input_b = torch.cat([context_b, suffix_ids]).unsqueeze(0).to(device)
    mask_a = torch.ones_like(input_a)
    mask_b = torch.ones_like(input_b)
    
    # Generate predictions (3 tokens)
    generated_a, generated_b = model.generate(
        input_ids_A=input_a,
        attention_mask_A=mask_a,
        input_ids_B=input_b,
        attention_mask_B=mask_b,
        max_new_tokens=3,
        temperature=0.7,
        do_sample=True,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id
    )
    
    # Decode generated tokens (only the new ones)
    input_len = input_a.shape[1]
    generated_tokens_a = generated_a[0, input_len:]
    generated_tokens_b = generated_b[0, input_len:]
    pred_text_a = tokenizer.decode(generated_tokens_a, skip_special_tokens=True).strip()
    pred_text_b = tokenizer.decode(generated_tokens_b, skip_special_tokens=True).strip()
    
    # Display results
    print("=" * 80)
    print(f"Text A: {tokenizer.convert_ids_to_tokens(input_a[0])}")
    print(f"Expected: {expected_a} | Predicted: '{pred_text_a}'")
    print()
    print(f"Text B: {tokenizer.convert_ids_to_tokens(input_b[0])}")
    print(f"Expected: {expected_b} | Predicted: '{pred_text_b}'")
    print("=" * 80)
    print()
    
    return pred_text_a, pred_text_b

print("Helper function defined. Use test_dual_classification() to test on new examples.")


Helper function defined. Use test_dual_classification() to test on new examples.


In [13]:
# Example 1: Music vs Technology
test_dual_classification(
    body_a="Text: The orchestra performed Beethoven's Symphony No. 9 with great passion.",
    body_b="Text: The new quantum computer processed complex calculations in milliseconds.",
    expected_a="Music",
    
    expected_b="Technology",
    prefix="Possible Categories: Music, Politics, Science, Sports, Technology, Travel.\n\n",
    suffix = '\nText Category: "'
)


Text A: ['Poss', 'ible', 'ĠC', 'ategories', ':', 'ĠMusic', ',', 'ĠPolitics', ',', 'ĠScience', ',', 'ĠSports', ',', 'ĠTechnology', ',', 'ĠTravel', '.', 'Ċ', 'Ċ', 'Text', ':', 'ĠThe', 'Ġorchestra', 'Ġperformed', 'ĠBe', 'eth', 'oven', "'s", 'ĠSymphony', 'ĠNo', '.', 'Ċ', 'Text', 'ĠCategory', ':', 'Ġ"']
Expected: Music | Predicted: 'Science, Technology'

Text B: ['Poss', 'ible', 'ĠC', 'ategories', ':', 'ĠMusic', ',', 'ĠPolitics', ',', 'ĠScience', ',', 'ĠSports', ',', 'ĠTechnology', ',', 'ĠTravel', '.', 'Ċ', 'Ċ', 'Text', ':', 'ĠThe', 'Ġnew', 'Ġquantum', 'Ġcomputer', 'Ġprocessed', 'Ġcomplex', 'Ġcalculations', 'Ġin', 'Ġmilliseconds', '.', 'Ċ', 'Text', 'ĠCategory', ':', 'Ġ"']
Expected: Technology | Predicted: 'C"'



('Science, Technology', 'C"')

In [14]:
# Example 2: History vs Cooking
test_dual_classification(
    body_a="Text: The Roman Empire fell in 476 AD, marking the end of ancient civilization.",
    body_b="Text: Mix flour, eggs, and milk to create a smooth batter for pancakes.",
    expected_a="History",
    expected_b="Cooking",
    prefix="Possible Categories: Art, Business, Cooking, History, Music, Politics, Science, Sports, Technology, Travel.\n\n",
    suffix = '\nText Category: "'
)


Text A: ['Poss', 'ible', 'ĠC', 'ategories', ':', 'ĠArt', ',', 'ĠBusiness', ',', 'ĠCook', 'ing', ',', 'ĠHistory', ',', 'ĠMusic', ',', 'ĠPolitics', ',', 'ĠScience', ',', 'ĠSports', ',', 'ĠTechnology', ',', 'ĠTravel', '.', 'Ċ', 'Ċ', 'Text', ':', 'ĠThe', 'ĠRoman', 'ĠEmpire', 'Ġfell', 'Ġin', 'Ġ476', 'ĠAD', ',', 'Ġmarking', 'Ġthe', 'Ġend', 'Ġof', 'Ġancient', 'Ġcivilization', '.', 'Ċ', 'Text', 'ĠCategory', ':', 'Ġ"']
Expected: History | Predicted: 'Warm".'

Text B: ['Poss', 'ible', 'ĠC', 'ategories', ':', 'ĠArt', ',', 'ĠBusiness', ',', 'ĠCook', 'ing', ',', 'ĠHistory', ',', 'ĠMusic', ',', 'ĠPolitics', ',', 'ĠScience', ',', 'ĠSports', ',', 'ĠTechnology', ',', 'ĠTravel', '.', 'Ċ', 'Ċ', 'Text', ':', 'ĠMix', 'Ġflour', ',', 'Ġeggs', ',', 'Ġand', 'Ġmilk', 'Ġto', 'Ġcreate', 'Ġa', 'Ġsmooth', 'Ġbatter', 'Ġfor', 'Ġpanc', 'akes', 'Ċ', 'Text', 'ĠCategory', ':', 'Ġ"']
Expected: Cooking | Predicted: 'Carmegg'



('Warm".', 'Carmegg')

In [15]:
# Example 3: Travel vs Politics
test_dual_classification(
    body_a="Text: The tropical beaches of Bali offer stunning sunsets and crystal clear water.",
    body_b="Text: The senator proposed a new bill to reform healthcare policy.",
    expected_a="Travel",
    expected_b="Politics",
    prefix="Possible Categories: Art, Business, Cooking, History, Music, Politics, Science, Sports, Technology, Travel.\n\n",
    suffix = '\nText Category: "'
)


Text A: ['Poss', 'ible', 'ĠC', 'ategories', ':', 'ĠArt', ',', 'ĠBusiness', ',', 'ĠCook', 'ing', ',', 'ĠHistory', ',', 'ĠMusic', ',', 'ĠPolitics', ',', 'ĠScience', ',', 'ĠSports', ',', 'ĠTechnology', ',', 'ĠTravel', '.', 'Ċ', 'Ċ', 'Text', ':', 'ĠThe', 'Ġtropical', 'Ġbeaches', 'Ġof', 'ĠB', 'ali', 'Ġoffer', 'Ġstunning', 'Ġsun', 'sets', 'Ġand', 'Ċ', 'Text', 'ĠCategory', ':', 'Ġ"']
Expected: Travel | Predicted: 'News"'

Text B: ['Poss', 'ible', 'ĠC', 'ategories', ':', 'ĠArt', ',', 'ĠBusiness', ',', 'ĠCook', 'ing', ',', 'ĠHistory', ',', 'ĠMusic', ',', 'ĠPolitics', ',', 'ĠScience', ',', 'ĠSports', ',', 'ĠTechnology', ',', 'ĠTravel', '.', 'Ċ', 'Ċ', 'Text', ':', 'ĠThe', 'Ġsenator', 'Ġproposed', 'Ġa', 'Ġnew', 'Ġbill', 'Ġto', 'Ġreform', 'Ġhealthcare', 'Ġpolicy', '.', 'Ċ', 'Text', 'ĠCategory', ':', 'Ġ"']
Expected: Politics | Predicted: 'Polit & Politics'



('News"', 'Polit & Politics')

## Perplexity and Token Rank Analysis

Detailed analysis of model predictions: token-by-token ranks and probabilities for expected sequences.


In [16]:
def analyze_perplexity_and_ranks(
    full_text_a,
    expected_text_a,
    full_text_b,
    expected_text_b,
    model,
    tokenizer,
    device
):
    """
    Analyze model predictions on expected text sequences.
    
    Args:
        full_text_a: Full text for stream A (context + expected)
        expected_text_a: Only the expected part for stream A
        full_text_b: Full text for stream B (context + expected)
        expected_text_b: Only the expected part for stream B
        model: The dual-stream model
        tokenizer: Tokenizer
        device: Device to run on
    
    Returns:
        Dictionary with analysis results for both streams
    """
    model.eval()
    
    # Tokenize full texts with offset mapping
    enc_a = tokenizer(full_text_a, return_offsets_mapping=True, return_tensors="pt")
    enc_b = tokenizer(full_text_b, return_offsets_mapping=True, return_tensors="pt")
    
    # Find where expected text starts in the full text (character position)
    expected_start_char_a = full_text_a.find(expected_text_a)
    expected_start_char_b = full_text_b.find(expected_text_b)
    
    if expected_start_char_a == -1:
        raise ValueError(f"Expected text A not found in full text A")
    if expected_start_char_b == -1:
        raise ValueError(f"Expected text B not found in full text B")
    
    # Find token indices where expected text starts
    expected_start_idx_a = find_split_token_index(enc_a, expected_start_char_a)
    expected_start_idx_b = find_split_token_index(enc_b, expected_start_char_b)
    
    # Calculate context and continuation lengths for both streams
    context_len_a = expected_start_idx_a
    context_len_b = expected_start_idx_b
    continuation_len_a = enc_a.input_ids.shape[1] - expected_start_idx_a
    continuation_len_b = enc_b.input_ids.shape[1] - expected_start_idx_b
    
    # Assert that context and continuation lengths are the same for both streams
    assert context_len_a == context_len_b, (
        f"Context length mismatch: A has {context_len_a} context tokens, "
        f"B has {context_len_b} context tokens"
    )
    assert continuation_len_a == continuation_len_b, (
        f"Continuation length mismatch: A has {continuation_len_a} continuation tokens, "
        f"B has {continuation_len_b} continuation tokens"
    )
    
    print(f"Expected text starts at token {expected_start_idx_a} for A, token {expected_start_idx_b} for B")
    print(f"Context: {context_len_a} tokens | Continuation: {continuation_len_a} tokens")
    
    # Prepare inputs (truncate to minimum length for alignment)
    min_len = min(enc_a.input_ids.shape[1], enc_b.input_ids.shape[1])
    input_ids_a = enc_a.input_ids[:, :min_len].to(device)
    input_ids_b = enc_b.input_ids[:, :min_len].to(device)
    attention_mask_a = torch.ones_like(input_ids_a)
    attention_mask_b = torch.ones_like(input_ids_b)
    
    # Forward pass
    with torch.no_grad():
        if model.objective == 'separate':
            logits_a, logits_b = model(input_ids_a, attention_mask_a, input_ids_b, attention_mask_b)
        else:
            logits = model(input_ids_a, attention_mask_a, input_ids_b, attention_mask_b)
            logits_a = logits
            logits_b = logits
    
    # Analyze stream A
    print("\n" + "="*80)
    print("STREAM A ANALYSIS")
    print("="*80)
    print(f"Full text: {full_text_a}")
    print(f"Expected starts at char {expected_start_char_a}, token {expected_start_idx_a}")
    print()
    
    results_a = _analyze_stream(
        input_ids_a, logits_a, expected_start_idx_a, min_len, tokenizer, "A"
    )
    
    # Analyze stream B
    print("\n" + "="*80)
    print("STREAM B ANALYSIS")
    print("="*80)
    print(f"Full text: {full_text_b}")
    print(f"Expected text: {expected_text_b}")
    print(f"Expected starts at char {expected_start_char_b}, token {expected_start_idx_b}")
    print()
    
    results_b = _analyze_stream(
        input_ids_b, logits_b, expected_start_idx_b, min_len, tokenizer, "B"
    )
    
    return {
        'stream_a': results_a,
        'stream_b': results_b
    }


def _analyze_stream(input_ids, logits, expected_start_idx, seq_len, tokenizer, stream_name):
    """Analyze predictions for a single stream."""
    
    # Extract expected tokens (from expected_start_idx to end)
    # Note: logits[:, i] predicts input_ids[:, i+1]
    # So logits at position i-1 predicts token at position i
    
    token_results = []
    log_probs_sum = 0.0
    num_tokens = 0
    
    print(f"{'Position':<10} {'Token':<25} {'Token ID':<10} {'Rank':<8} {'Probability':<12} {'Log Prob':<12}")
    print("-" * 90)
    
    for pos in range(expected_start_idx, seq_len):
        # Token at position `pos`
        true_token_id = input_ids[0, pos].item()
        true_token = tokenizer.decode([true_token_id])
        
        # Logits at position pos-1 predict token at position pos
        if pos == 0:
            continue  # No prediction for first token
        
        pred_logits = logits[0, pos - 1, :]  # [vocab_size]
        probs = torch.softmax(pred_logits, dim=-1)
        
        # Get probability of true token
        true_prob = probs[true_token_id].item()
        true_log_prob = torch.log(probs[true_token_id]).item()
        
        # Get rank of true token
        sorted_probs, sorted_indices = torch.sort(probs, descending=True)
        rank = (sorted_indices == true_token_id).nonzero(as_tuple=True)[0].item() + 1
        
        token_results.append({
            'position': pos,
            'token': true_token,
            'token_id': true_token_id,
            'rank': rank,
            'probability': true_prob,
            'log_prob': true_log_prob
        })
        
        log_probs_sum += true_log_prob
        num_tokens += 1
        
        print(f"{pos:<10} {repr(true_token):<25} {true_token_id:<10} {rank:<8} {true_prob:<12.6f} {true_log_prob:<12.4f}")
    
    # Calculate perplexity
    if num_tokens > 0:
        avg_log_prob = log_probs_sum / num_tokens
        perplexity = torch.exp(torch.tensor(-avg_log_prob)).item()
    else:
        perplexity = float('inf')
    
    print()
    print(f"{'='*90}")
    print(f"STREAM {stream_name} SUMMARY:")
    print(f"  Total expected tokens: {num_tokens}")
    print(f"  Average log probability: {avg_log_prob:.4f}" if num_tokens > 0 else "  Average log probability: N/A")
    print(f"  Perplexity: {perplexity:.4f}")
    print(f"{'='*90}")
    
    return {
        'tokens': token_results,
        'perplexity': perplexity,
        'num_tokens': num_tokens,
        'avg_log_prob': avg_log_prob if num_tokens > 0 else None
    }

print("✓ Perplexity analysis functions defined")


✓ Perplexity analysis functions defined


In [17]:
# Example: Analyze number sequence prediction
full_text_a = "300 400 500 600 700 800"
expected_text_a = "800"

full_text_b = "1 2 3 4 5 6"
expected_text_b = "6"

results = analyze_perplexity_and_ranks(
    full_text_a=full_text_a,
    expected_text_a=expected_text_a,
    full_text_b=full_text_b,
    expected_text_b=expected_text_b,
    model=model,
    tokenizer=tokenizer,
    device=device
)


Expected text starts at token 5 for A, token 5 for B
Context: 5 tokens | Continuation: 1 tokens

STREAM A ANALYSIS
Full text: 300 400 500 600 700 800
Expected starts at char 20, token 5

Position   Token                     Token ID   Rank     Probability  Log Prob    
------------------------------------------------------------------------------------------


5          ' 800'                    14212      15       0.011719     -4.4375     

STREAM A SUMMARY:
  Total expected tokens: 1
  Average log probability: -4.4375
  Perplexity: 84.5633

STREAM B ANALYSIS
Full text: 1 2 3 4 5 6
Expected text: 6
Expected starts at char 10, token 5

Position   Token                     Token ID   Rank     Probability  Log Prob    
------------------------------------------------------------------------------------------
5          ' 6'                      721        1        0.443359     -0.8125     

STREAM B SUMMARY:
  Total expected tokens: 1
  Average log probability: -0.8125
  Perplexity: 2.2535


In [18]:
full_text_a = 'Possible Categories: Art, Business, Cooking, History, Music, Politics, Science, Sports, Technology, Travel.\n\nText: The astronomers observed a massive black hole in a distant galaxy.\nText Category: "Science"'
expected_text_a = '"Science"'

full_text_b = 'Possible Categories: Art, Business, Cooking, History, Music, Politics, Science, Sports, Technology, Travel.\n\nText: The striker kicked the ball into the net to win the match.\nText Category: "Sports"'
expected_text_b = '"Sports"'


results = analyze_perplexity_and_ranks(
    full_text_a=full_text_a,
    expected_text_a=expected_text_a,
    full_text_b=full_text_b,
    expected_text_b=expected_text_b,
    model=model,
    tokenizer=tokenizer,
    device=device
)


Expected text starts at token 47 for A, token 47 for B
Context: 47 tokens | Continuation: 3 tokens

STREAM A ANALYSIS
Full text: Possible Categories: Art, Business, Cooking, History, Music, Politics, Science, Sports, Technology, Travel.

Text: The astronomers observed a massive black hole in a distant galaxy.
Text Category: "Science"
Expected starts at char 197, token 47

Position   Token                     Token ID   Rank     Probability  Log Prob    
------------------------------------------------------------------------------------------
47         ' "'                      346        112      0.000927     -6.9688     
48         'Science'                 34191      6        0.015747     -4.1562     
49         '"'                       3          1        0.566406     -0.5703     

STREAM A SUMMARY:
  Total expected tokens: 3
  Average log probability: -3.8984
  Perplexity: 49.3253

STREAM B ANALYSIS
Full text: Possible Categories: Art, Business, Cooking, History, Music, Politics

In [19]:
# Example 2: Analyze text classification task
prefix = "Possible Categories: Art, Business, Cooking, History, Music, Politics, Science, Sports, Technology, Travel.\n\n"
suffix = '\nText Category: "'

body_a = "Text: The astronomers observed a massive black hole in a distant galaxy."
body_b = "Text: The striker kicked the ball into the net to win the match"

expected_a = "Science"
expected_b = "Sports"

full_text_a = prefix + body_a + suffix + expected_a + '"'
full_text_b = prefix + body_b + suffix + expected_b + '"'

results = analyze_perplexity_and_ranks(
    full_text_a=full_text_a,
    expected_text_a=expected_a,
    full_text_b=full_text_b,
    expected_text_b=expected_b,
    model=model,
    tokenizer=tokenizer,
    device=device
)


AssertionError: Context length mismatch: A has 18 context tokens, B has 20 context tokens